# Stage 1: Data Preprocessing & Feature Extraction
In this notebook, we handle the first stages of the Spatio-Temporal SAR Architecture:
1. **Data Input**: Loading SAR Image Sequences from the Sen1Floods11 dataset.
2. **Preprocessing**: Radiometric calibration (clipping), Normalization, and **Speckle Noise Reduction**.
3. **Feature Extraction**: Training a Convolutional Autoencoder (CAE) to capture important spatial features.
The trained Encoder will be saved for Stage 2 (Temporal Modeling).

In [1]:
import os
import urllib.request
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import uniform_filter

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### 1. Data Input (Sen1Floods11)
Downloading a subset of the data for training and testing.

In [2]:
DATA_DIR = "data/sen1floods11"
S1_DIR = os.path.join(DATA_DIR, "S1Hand")
LABEL_DIR = os.path.join(DATA_DIR, "LabelHand")
os.makedirs(S1_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

BASE_URL = "https://storage.googleapis.com/sen1floods11/v1.1"
SPLIT_BASE_URL = f"{BASE_URL}/splits/flood_handlabeled"
DATA_BASE_URL = f"{BASE_URL}/data/flood_events/HandLabeled"

train_csv_path = os.path.join(DATA_DIR, "flood_train_data.csv")
test_csv_path = os.path.join(DATA_DIR, "flood_test_data.csv")

if not os.path.exists(train_csv_path):
    urllib.request.urlretrieve(f"{SPLIT_BASE_URL}/flood_train_data.csv", train_csv_path)

if not os.path.exists(test_csv_path):
    urllib.request.urlretrieve(f"{SPLIT_BASE_URL}/flood_test_data.csv", test_csv_path)

def download_subset(csv_path, num_samples):
    downloaded_files = []
    with open(csv_path, "r") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i >= num_samples:
                break
            s1_file, label_file = row[0], row[1]
            s1_local = os.path.join(S1_DIR, s1_file)
            label_local = os.path.join(LABEL_DIR, label_file)
            
            if not os.path.exists(s1_local):
                urllib.request.urlretrieve(f"{DATA_BASE_URL}/S1Hand/{s1_file}", s1_local)
            if not os.path.exists(label_local):
                urllib.request.urlretrieve(f"{DATA_BASE_URL}/LabelHand/{label_file}", label_local)
                
            downloaded_files.append((s1_local, label_local))
    return downloaded_files

# Download a small subset for demonstration
print("Downloading data...")
train_files = download_subset(train_csv_path, 30)
test_files = download_subset(test_csv_path, 10)
print(f"Downloaded {len(train_files)} train and {len(test_files)} test pairs.")

Downloaded 30 train and 10 test pairs.


### 2. Preprocessing & Speckle Noise Reduction
Here we implement:
- **Speckle Noise Reduction**: A simple mean/box filter applied to the SAR data.
- **Radiometric Calibration & Normalization**: Clipping to valid dB ranges and scaling to [0, 1].

In [3]:
def apply_speckle_filter(image, size=3):
    """Applies a simple box filter to reduce speckle noise."""
    filtered_image = np.zeros_like(image)
    for band in range(image.shape[0]):
        filtered_image[band] = uniform_filter(image[band], size=size)
    return filtered_image

def preprocess_s1(data):
    """Fill NaNs, apply speckle filter, clip range, and normalize."""
    # Fill NaNs
    vv = np.nan_to_num(data[0], nan=-25.0)
    vh = np.nan_to_num(data[1], nan=-30.0)
    
    # Apply Speckle Filter (on dB scale)
    stacked = np.stack([vv, vh], axis=0)
    filtered = apply_speckle_filter(stacked, size=3)
    
    vv_f, vh_f = filtered[0], filtered[1]
    
    # Clip (Radiometric Calibration)
    vv_clipped = np.clip(vv_f, -30.0, 0.0)
    vh_clipped = np.clip(vh_f, -35.0, -5.0)
    
    # Normalize to [0, 1]
    vv_norm = (vv_clipped - (-30.0)) / 30.0
    vh_norm = (vh_clipped - (-35.0)) / 30.0
    
    return np.stack([vv_norm, vh_norm], axis=0)

class Sen1FloodsDataset(Dataset):
    def __init__(self, file_pairs, target_size=(256, 256)):
        self.file_pairs = file_pairs
        self.target_size = target_size
        
    def __len__(self):
        return len(self.file_pairs)
        
    def __getitem__(self, idx):
        s1_path, label_path = self.file_pairs[idx]
        
        with rasterio.open(s1_path) as src:
            s1_data = src.read()
        
        with rasterio.open(label_path) as src:
            label_data = src.read(1)
            
        s1_preprocessed = preprocess_s1(s1_data)
        
        s1_tensor = torch.tensor(s1_preprocessed, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(label_data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
        s1_resized = torch.nn.functional.interpolate(
            s1_tensor, size=self.target_size, mode='bilinear', align_corners=False
        ).squeeze(0)
        
        label_resized = torch.nn.functional.interpolate(
            label_tensor, size=self.target_size, mode='nearest'
        ).squeeze(0).squeeze(0).long()
        
        return s1_resized, label_resized

train_dataset = Sen1FloodsDataset(train_files)
test_dataset = Sen1FloodsDataset(test_files)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)


### 3. Feature Extraction (Autoencoder)
We train a Convolutional Autoencoder. The encoder part will learn to compress the preprocessed SAR imagery into a Latent Representation.

In [4]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        # Encoder: 256x256x2 -> 16x16x128
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True)
        )
        # Decoder: 16x16x128 -> 256x256x2
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 2, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

model = ConvAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
print("Starting Autoencoder training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for data, _ in train_loader:
        data = data.to(device)
        reconstructed, _ = model(data)
        loss = criterion(reconstructed, data)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.5f}")

print("Training Complete!")

Starting Autoencoder training...
Epoch [1/10], Loss: 0.03374
Epoch [2/10], Loss: 0.02521
Epoch [3/10], Loss: 0.01727
Epoch [4/10], Loss: 0.01374
Epoch [5/10], Loss: 0.01207
Epoch [6/10], Loss: 0.01068
Epoch [7/10], Loss: 0.01051
Epoch [8/10], Loss: 0.01082
Epoch [9/10], Loss: 0.01079
Epoch [10/10], Loss: 0.01074
Training Complete!


### 4. Save Encoder Weights
We save the trained Encoder weights. These will be loaded in Stage 2 to provide the latent feature sequences to the Temporal Modeling Transformer.

In [5]:
os.makedirs('models', exist_ok=True)
torch.save(model.encoder.state_dict(), 'models/encoder.pth')
print("Encoder weights saved to models/encoder.pth")

Encoder weights saved to models/encoder.pth
